In [1]:
import sys
import pandas as pd

print("Intérprete:", sys.executable)
print("Versión de pandas:", pd.__version__)

Intérprete: c:\Users\Manu\anaconda3\envs\Netflix_Cat_2021\python.exe
Versión de pandas: 3.0.5


In [2]:
from pathlib import Path

raiz_proyecto = Path.cwd()

if raiz_proyecto.name == "notebooks":
    raiz_proyecto = raiz_proyecto.parent

ruta_datos = raiz_proyecto / "archive" / "netflixData.csv"

print("Carpeta del proyecto:", raiz_proyecto)
print("Archivo encontrado:", ruta_datos.exists())
print("Ruta:", ruta_datos)

Carpeta del proyecto: c:\Users\Manu\Desktop\Master\Proyectos_Protfolio\Netflix_Catalogo_2021
Archivo encontrado: True
Ruta: c:\Users\Manu\Desktop\Master\Proyectos_Protfolio\Netflix_Catalogo_2021\archive\netflixData.csv


In [3]:
df = pd.read_csv(ruta_datos)

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

df.head()

Filas: 5967
Columnas: 13


,Show Id,Title,Description,Director,Genres,Cast,Production Country,Release Date,Rating,Duration,Imdb Score,Content Type,Date Added
0,cc1b6ed9-cf9e-4057-8303-34577fb54477,(Un)Well,This docuseries takes a deep dive into the luc...,NaN,Reality TV,NaN,United States,2020.0,TV-MA,1 Season,6.6/10,TV Show,NaN
1,e2ef4e91-fb25-42ab-b485-be8e3b23dedb,#Alive,"As a grisly virus rampages a city, a lone man ...",Cho Il,"Horror Movies, International Movies, Thrillers","Yoo Ah-in, Park Shin-hye",South Korea,2020.0,TV-MA,99 min,6.2/10,Movie,"September 8, 2020"
2,b01b73b7-81f6-47a7-86d8-acb63080d525,#AnneFrank - Parallel Stories,"Through her diary, Anne Frank's story is retol...","Sabina Fedeli, Anna Migotto","Documentaries, International Movies","Helen Mirren, Gengher Gatti",Italy,2019.0,TV-14,95 min,6.4/10,Movie,"July 1, 2020"
3,b6611af0-f53c-4a08-9ffa-9716dc57eb9c,#blackAF,Kenya Barris and his family navigate relations...,NaN,TV Comedies,"Kenya Barris, Rashida Jones, Iman Benson, Genn...",United States,2020.0,TV-MA,1 Season,6.6/10,TV Show,NaN
4,7f2d4170-bab8-4d75-adc2-197f7124c070,#cats_the_mewvie,This pawesome documentary explores how our fel...,Michael Margolis,"Documentaries, International Movies",NaN,Canada,2020.0,TV-14,90 min,5.1/10,Movie,"February 5, 2020"


In [4]:
resumen_columnas = pd.DataFrame({
    "tipo_dato": df.dtypes.astype(str),
    "valores_nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique()
})

resumen_columnas.sort_values(
    by="porcentaje_nulos",
    ascending=False
)

,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos
Director,str,2064,34.59,2993
Date Added,str,1335,22.37,1283
Imdb Score,str,608,10.19,79
Production Country,str,559,9.37,509
Cast,str,530,8.88,5245
Rating,str,4,0.07,11
Release Date,float64,3,0.05,65
Duration,str,3,0.05,207
Show Id,str,0,0.00,5967
Genres,str,0,0.00,433


In [5]:
filas_duplicadas = df.duplicated().sum()
titulos_repetidos = df.duplicated(subset=["Title"]).sum()

print("Filas completamente duplicadas:", filas_duplicadas)
print("Títulos repetidos:", titulos_repetidos)

Filas completamente duplicadas: 0
Títulos repetidos: 70


In [7]:
# Creamos una copia para conservar intactos los datos originales
df_limpio = df.copy()

# Renombramos la columna
df_limpio = df_limpio.rename(
    columns={"Release Date": "Release Year"}
)

# Quitamos los decimales conservando los valores nulos
df_limpio["Release Year"] = (
    pd.to_numeric(
        df_limpio["Release Year"],
        errors="coerce"
    )
    .astype("Int64")
)

# Columnas que queremos mostrar para comparar títulos repetidos
columnas_comparacion = [
    "Title",
    "Content Type",
    "Release Year",
    "Production Country"
]

# Mostramos algunos títulos repetidos
df_limpio[
    df_limpio.duplicated(subset=["Title"], keep=False)
][columnas_comparacion].sort_values("Title").head(20)

,Title,Content Type,Release Year,Production Country
124,A Love So Beautiful,TV Show,2017,China
125,A Love So Beautiful,TV Show,2020,South Korea
254,Alaska Is a Drag,Movie,2017,United States
255,Alaska Is a Drag,Movie,2017,United States
283,All About Love,TV Show,2017,Taiwan
284,All About Love,Movie,2017,South Africa
311,Always Be My Maybe,Movie,2016,Philippines
312,Always Be My Maybe,Movie,2019,United States
452,Aurora,Movie,2018,Philippines
453,Aurora,Movie,2010,"Romania, France, Switzerland, Germany"


In [8]:
# Eliminamos "/10" y convertimos la puntuación en un número
df_limpio["Imdb Score"] = pd.to_numeric(
    df_limpio["Imdb Score"].str.replace(
        "/10",
        "",
        regex=False
    ),
    errors="coerce"
)

# Comprobamos el resultado
print("Tipo de dato:", df_limpio["Imdb Score"].dtype)

df_limpio[["Title", "Imdb Score"]].head()

Tipo de dato: float64


,Title,Imdb Score
0,(Un)Well,6.6
1,#Alive,6.2
2,#AnneFrank - Parallel Stories,6.4
3,#blackAF,6.6
4,#cats_the_mewvie,5.1


In [9]:
# Columnas de texto que utilizará el recomendador
columnas_texto = [
    "Description",
    "Genres",
    "Director",
    "Cast",
    "Production Country"
]

# Sustituimos los valores ausentes por texto vacío
df_limpio[columnas_texto] = (
    df_limpio[columnas_texto]
    .fillna("")
)

# Comprobamos que ya no contienen valores nulos
df_limpio[columnas_texto].isna().sum()

Description           0
Genres                0
Director              0
Cast                  0
Production Country    0
dtype: int64

In [10]:
def normalizar_elementos(texto):
    """
    Convierte una lista separada por comas en términos normalizados.
    
    Ejemplo:
    'Horror Movies, International Movies'
    se convierte en:
    'horror_movies international_movies'
    """
    
    elementos = texto.split(",")
    
    elementos_normalizados = [
        elemento.strip().lower().replace(" ", "_")
        for elemento in elementos
        if elemento.strip()
    ]
    
    return " ".join(elementos_normalizados)


# Creamos columnas preparadas para el modelo
df_limpio["Genres Model"] = (
    df_limpio["Genres"].apply(normalizar_elementos)
)

df_limpio["Director Model"] = (
    df_limpio["Director"].apply(normalizar_elementos)
)

df_limpio["Cast Model"] = (
    df_limpio["Cast"].apply(normalizar_elementos)
)

df_limpio["Country Model"] = (
    df_limpio["Production Country"].apply(normalizar_elementos)
)

# Comprobamos el resultado
df_limpio[
    [
        "Genres",
        "Genres Model",
        "Director",
        "Director Model"
    ]
].head()

,Genres,Genres Model,Director,Director Model
0,Reality TV,reality_tv,,
1,"Horror Movies, International Movies, Thrillers",horror_movies international_movies thrillers,Cho Il,cho_il
2,"Documentaries, International Movies",documentaries international_movies,"Sabina Fedeli, Anna Migotto",sabina_fedeli anna_migotto
3,TV Comedies,tv_comedies,,
4,"Documentaries, International Movies",documentaries international_movies,Michael Margolis,michael_margolis


In [11]:
# Normalizamos la descripción
df_limpio["Description Model"] = (
    df_limpio["Description"]
    .fillna("")
    .str.lower()
    .str.strip()
)

# Combinamos las características y aplicamos pesos
df_limpio["Combined Features"] = (
    df_limpio["Genres Model"] + " " +
    df_limpio["Genres Model"] + " " +
    df_limpio["Genres Model"] + " " +
    df_limpio["Director Model"] + " " +
    df_limpio["Director Model"] + " " +
    df_limpio["Cast Model"] + " " +
    df_limpio["Country Model"] + " " +
    df_limpio["Description Model"]
).str.strip()

# Comprobamos el resultado
df_limpio[
    ["Title", "Combined Features"]
].head()

,Title,Combined Features
0,(Un)Well,reality_tv reality_tv reality_tv united_sta...
1,#Alive,horror_movies international_movies thrillers h...
2,#AnneFrank - Parallel Stories,documentaries international_movies documentari...
3,#blackAF,tv_comedies tv_comedies tv_comedies kenya_ba...
4,#cats_the_mewvie,documentaries international_movies documentari...


## Vectorización con TF-IDF

Convertimos las características textuales en vectores numéricos.
TF-IDF aumenta la importancia de los términos representativos y 
reduce el peso de las palabras demasiado frecuentes.

In [ ]:
# Herramienta de scikit-learn para convertir texto en vectores numéricos
from sklearn.feature_extraction.text import TfidfVectorizer

# Configuramos el vectorizador
vectorizador = TfidfVectorizer(
    # Elimina palabras inglesas poco informativas como "the" o "and"
    stop_words="english",
    
    # Analiza palabras individuales y parejas de palabras
    ngram_range=(1, 2),
    
    # Limita el vocabulario para controlar memoria y tiempo
    max_features=20_000
)


# Aprende el vocabulario y transforma cada título en un vector
matriz_tfidf = vectorizador.fit_transform(
    df_limpio["Combined Features"]
)


# Comprobamos las dimensiones obtenidas
print("Dimensiones de la matriz:", matriz_tfidf.shape)

print(
    "Características utilizadas:",
    len(vectorizador.get_feature_names_out())
)

Dimensiones de la matriz: (5967, 20000)
Características utilizadas: 20000


## Motor de recomendación

Calculamos la similitud del coseno entre un título seleccionado y el resto del catálogo. Después devolvemos los títulos con mayor similitud.

In [15]:
from sklearn.metrics.pairwise import cosine_similarity


def recomendar_por_indice(indice_titulo, cantidad=5):
    """
    Devuelve los títulos más similares al registro seleccionado.

    Parámetros:
        indice_titulo: posición del título en el DataFrame.
        cantidad: número de recomendaciones.
    """

    # Comparamos un título con todo el catálogo
    similitudes = cosine_similarity(
        matriz_tfidf[indice_titulo],
        matriz_tfidf
    ).flatten()

    # Ordenamos los índices desde la similitud más alta
    indices_ordenados = similitudes.argsort()[::-1]

    # Excluimos el propio título y conservamos la cantidad solicitada
    indices_recomendados = indices_ordenados[
        indices_ordenados != indice_titulo
    ][:cantidad]

    # Información que mostraremos de cada recomendación
    columnas_resultado = [
        "Title",
        "Content Type",
        "Release Year",
        "Genres",
        "Imdb Score"
    ]

    # Recuperamos los títulos recomendados
    recomendaciones = (
        df_limpio
        .iloc[indices_recomendados][columnas_resultado]
        .copy()
    )

    # Convertimos la similitud en un porcentaje más legible
    recomendaciones["Similarity (%)"] = (
        similitudes[indices_recomendados] * 100
    ).round(2)

    return recomendaciones

In [16]:
indice_prueba = 1

print(
    "Título seleccionado:",
    df_limpio.iloc[indice_prueba]["Title"]
)

recomendar_por_indice(
    indice_titulo=indice_prueba,
    cantidad=5
)

Título seleccionado: #Alive


,Title,Content Type,Release Year,Genres,Imdb Score,Similarity (%)
3629,Paranormal Investigation,Movie,2018,"Horror Movies, International Movies, Thrillers",3.6,55.20
2104,HOMUNCULUS,Movie,2021,"Horror Movies, International Movies, Thrillers",5.5,55.19
4658,The Binding,Movie,2020,"Horror Movies, International Movies, Thrillers",4.8,53.73
5168,The Ritual,Movie,2018,"Horror Movies, International Movies, Thrillers",6.2,52.93
5656,Veronica,Movie,2017,"Horror Movies, International Movies, Thrillers",6.1,52.55


In [17]:
# Convertimos el año en texto para construir la etiqueta
anio_texto = (
    df_limpio["Release Year"]
    .astype("string")
    .fillna("Año desconocido")
)

# Creamos una etiqueta fácil de identificar
df_limpio["Selection Label"] = (
    df_limpio["Title"] +
    " (" +
    anio_texto +
    ", " +
    df_limpio["Content Type"] +
    ")"
)

# Detectamos etiquetas que todavía están repetidas
etiquetas_duplicadas = (
    df_limpio["Selection Label"]
    .duplicated(keep=False)
)

# Añadimos parte del identificador únicamente a las repetidas
df_limpio.loc[
    etiquetas_duplicadas,
    "Selection Label"
] = (
    df_limpio.loc[
        etiquetas_duplicadas,
        "Selection Label"
    ] +
    " [" +
    df_limpio.loc[
        etiquetas_duplicadas,
        "Show Id"
    ].str[:8] +
    "]"
)

# Relacionamos cada etiqueta con su posición en la matriz
indice_por_etiqueta = pd.Series(
    range(len(df_limpio)),
    index=df_limpio["Selection Label"]
)

df_limpio[
    ["Title", "Selection Label"]
].head()

,Title,Selection Label
0,(Un)Well,"(Un)Well (2020, TV Show)"
1,#Alive,"#Alive (2020, Movie)"
2,#AnneFrank - Parallel Stories,"#AnneFrank - Parallel Stories (2019, Movie)"
3,#blackAF,"#blackAF (2020, TV Show)"
4,#cats_the_mewvie,"#cats_the_mewvie (2020, Movie)"


In [18]:
def recomendar_por_titulo(etiqueta_titulo, cantidad=5):
    """
    Recomienda contenidos utilizando una etiqueta legible.
    """

    if etiqueta_titulo not in indice_por_etiqueta:
        raise ValueError(
            f"No se encontró el título: {etiqueta_titulo}"
        )

    indice_titulo = int(
        indice_por_etiqueta.loc[etiqueta_titulo]
    )

    return recomendar_por_indice(
        indice_titulo=indice_titulo,
        cantidad=cantidad
    )

In [19]:
titulo_prueba = df_limpio.iloc[1]["Selection Label"]

print("Título seleccionado:", titulo_prueba)

recomendar_por_titulo(
    etiqueta_titulo=titulo_prueba,
    cantidad=5
)

Título seleccionado: #Alive (2020, Movie)


,Title,Content Type,Release Year,Genres,Imdb Score,Similarity (%)
3629,Paranormal Investigation,Movie,2018,"Horror Movies, International Movies, Thrillers",3.6,55.20
2104,HOMUNCULUS,Movie,2021,"Horror Movies, International Movies, Thrillers",5.5,55.19
4658,The Binding,Movie,2020,"Horror Movies, International Movies, Thrillers",4.8,53.73
5168,The Ritual,Movie,2018,"Horror Movies, International Movies, Thrillers",6.2,52.93
5656,Veronica,Movie,2017,"Horror Movies, International Movies, Thrillers",6.1,52.55


In [20]:
def buscar_titulos(texto, cantidad=10):
    """
    Busca títulos que contengan el texto indicado.
    """

    coincidencias = df_limpio["Title"].str.contains(
        texto,
        case=False,
        na=False,
        regex=False
    )

    columnas_resultado = [
        "Selection Label",
        "Genres",
        "Imdb Score"
    ]

    return (
        df_limpio
        .loc[coincidencias, columnas_resultado]
        .head(cantidad)
    )

In [21]:
buscar_titulos(
    texto="Stranger",
    cantidad=10
)

,Selection Label,Genres,Imdb Score
639,"Beyond Stranger Things (2017, TV Show)","Stand-Up Comedy & Talk Shows, TV Mysteries, TV...",7.6
4436,"Stranger (2020, TV Show)","Crime TV Shows, International TV Shows, Korean...",8.6
4437,"Stranger than Fiction (2006, Movie)","Comedies, Romantic Movies",7.6
4438,"Stranger Things (2019, TV Show)","TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",8.7
4439,"Strangers from Hell (2019, TV Show)","International TV Shows, Korean TV Shows, TV Ho...",8.0
5228,"THE STRANGER (2020, TV Show)","British TV Shows, Crime TV Shows, Internationa...",7.2
5229,"The Strangers (2008, Movie)","Horror Movies, Thrillers",6.0
5230,"The Strangers: Prey at Night (2018, Movie)",Horror Movies,5.3
5549,"Two Distant Strangers (2021, Movie)",Dramas,7.0


In [22]:
# Localizamos la etiqueta exacta de Stranger Things
titulo_stranger_things = df_limpio.loc[
    df_limpio["Title"].eq("Stranger Things"),
    "Selection Label"
].iloc[0]

print(
    "Título seleccionado:",
    titulo_stranger_things
)

# Generamos las recomendaciones
recomendar_por_titulo(
    etiqueta_titulo=titulo_stranger_things,
    cantidad=5
)

Título seleccionado: Stranger Things (2019, TV Show)


,Title,Content Type,Release Year,Genres,Imdb Score,Similarity (%)
3406,Nightflyers,TV Show,2018,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",6.0,64.29
978,Chilling Adventures of Sabrina,TV Show,2020,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",7.4,62.19
639,Beyond Stranger Things,TV Show,2017,"Stand-Up Comedy & Talk Shows, TV Mysteries, TV...",7.6,54.05
2941,Manifest,TV Show,2020,"TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy",7.1,48.56
4605,The 4400,TV Show,2007,"TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy",7.4,46.19


### Evaluación inicial

El recomendador devuelve resultados coherentes para *Stranger Things*, principalmente series de terror, misterio y ciencia ficción.

También recomienda *Beyond Stranger Things*, un programa relacionado directamente con la serie. Esto demuestra que el modelo detecta relaciones temáticas, pero también revela que será necesario permitir filtros por género y tipo de contenido.

La puntuación de similitud representa proximidad entre las características textuales, no la probabilidad de que una persona disfrute del contenido.